<a href="https://colab.research.google.com/github/vandit-11/learning-data-science/blob/main/spark_crash_course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
  .appName("SparkInterviewPrep") \
  .getOrCreate()

In [4]:
print("Spark version: ", spark.version)

Spark version:  4.0.2


In [5]:
data = [("Alice", "HR", 5000),
        ("Bob", "IT", 6000),
        ("Charlie", "IT", 6500)]

columns = ["employee_name", "department", "salary"]

In [6]:
# Combine the data and columns into a Spark "DataFrame"
df = spark.createDataFrame(data, columns)

df.show(1)

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|        Alice|        HR|  5000|
+-------------+----------+------+
only showing top 1 row


In [7]:
df.select("employee_name", "salary").show(2)

+-------------+------+
|employee_name|salary|
+-------------+------+
|        Alice|  5000|
|          Bob|  6000|
+-------------+------+
only showing top 2 rows


In [10]:
df.filter(df.department == "IT").show()

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|          Bob|        IT|  6000|
|      Charlie|        IT|  6500|
+-------------+----------+------+



In [11]:
df.filter(df.salary > 5500).select("employee_name", "salary").show()

+-------------+------+
|employee_name|salary|
+-------------+------+
|          Bob|  6000|
|      Charlie|  6500|
+-------------+------+



In [12]:
data2 =[("Alice", "HR", 50000),
         ("Bob", "IT", 60000),
         ("Charlie", "IT", 65000),
         ("Diana", "HR", 55000),
         ("Evan", "Sales", 70000),
         ("Fiona", "Sales", 72000)]

In [13]:
df2 = spark.createDataFrame(data2, ["employee_name", "department", "salary"])
df2.show()

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|        Alice|        HR| 50000|
|          Bob|        IT| 60000|
|      Charlie|        IT| 65000|
|        Diana|        HR| 55000|
|         Evan|     Sales| 70000|
|        Fiona|     Sales| 72000|
+-------------+----------+------+



In [14]:
df2.groupby("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    2|
|        IT|    2|
|     Sales|    2|
+----------+-----+



In [15]:
df2.groupby("department").sum("salary").show()

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        HR|     105000|
|        IT|     125000|
|     Sales|     142000|
+----------+-----------+



#Spark SQL
Earlier, we were only using DataFrame API. But spark has a secret weapon where we can register our dataframe as a temporary SQL table.

In [17]:
# Create a temp view of the df2
df2.createOrReplaceTempView("employees")

In [24]:
# Write basic sql inside now...
sql_df = spark.sql("select department, salary from employees where salary > 60000")
sql_df.show()

+----------+------+
|department|salary|
+----------+------+
|        IT| 65000|
|     Sales| 70000|
|     Sales| 72000|
+----------+------+



In [27]:
sql_hw = spark.sql("select department, avg(salary) from employees group by department").cache()

In [29]:
sql_hw.show()

+----------+-----------+
|department|avg(salary)|
+----------+-----------+
|     Sales|    71000.0|
|        HR|    52500.0|
|        IT|    62500.0|
+----------+-----------+



In [30]:
# 1. Check current partitions of our original df2
print("Original Partitions:", df2.rdd.getNumPartitions())

# 2. Let's increase partitions to 8 using repartition()
df_repartitioned = df2.repartition(8)
print("After Repartition(8):", df_repartitioned.rdd.getNumPartitions())

# 3. Now let's decrease them back to 2 using coalesce()
df_coalesced = df_repartitioned.coalesce(2)
print("After Coalesce(2):", df_coalesced.rdd.getNumPartitions())

Original Partitions: 2
After Repartition(8): 8
After Coalesce(2): 2


# Joins

In [31]:
from pyspark.sql.functions import broadcast

In [32]:
dept_data =[("HR", "New York"),
             ("IT", "San Francisco"),
             ("Sales", "Chicago")]

In [33]:
dept_df = spark.createDataFrame(dept_data, ["department", "location"])
dept_df.show()

+----------+-------------+
|department|     location|
+----------+-------------+
|        HR|     New York|
|        IT|San Francisco|
|     Sales|      Chicago|
+----------+-------------+



In [34]:
standard_join = df2.join(dept_df, on="department", how="inner")
standard_join.show()

+----------+-------------+------+-------------+
|department|employee_name|salary|     location|
+----------+-------------+------+-------------+
|        HR|        Alice| 50000|     New York|
|        HR|        Diana| 55000|     New York|
|        IT|          Bob| 60000|San Francisco|
|        IT|      Charlie| 65000|San Francisco|
|     Sales|         Evan| 70000|      Chicago|
|     Sales|        Fiona| 72000|      Chicago|
+----------+-------------+------+-------------+



In [35]:
broadcast_join = df2.join(broadcast(dept_df), on="department", how="inner")
broadcast_join.show()

+----------+-------------+------+-------------+
|department|employee_name|salary|     location|
+----------+-------------+------+-------------+
|        HR|        Alice| 50000|     New York|
|        IT|          Bob| 60000|San Francisco|
|        IT|      Charlie| 65000|San Francisco|
|        HR|        Diana| 55000|     New York|
|     Sales|         Evan| 70000|      Chicago|
|     Sales|        Fiona| 72000|      Chicago|
+----------+-------------+------+-------------+

